In [8]:
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from qdrant_client.models import Distance, VectorParams

In [2]:
client = QdrantClient(url="http://localhost:6333")

In [4]:
client.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.DOT),
)

True

In [19]:
random_embeddings = np.random.normal(size=(10000, 1024))
random_embeddings /= np.sqrt((random_embeddings**2).sum(axis=-1))[:, None]

In [21]:
all_points = [PointStruct(id=l, vector = embed, payload={"city": "Berlin"}) for l, embed in enumerate(random_embeddings)]

In [22]:
operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=all_points[:1000],
)


In [27]:
query = np.random.normal(size=(1024,))
query = query/np.sqrt((query**2).sum())

In [28]:
search_result = client.query_points(
    collection_name="test_collection",
    query=query,
    with_payload=False,
    limit=3
).points

In [29]:
search_result

[ScoredPoint(id=845, version=2, score=0.10005581, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=791, version=2, score=0.08682226, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=810, version=2, score=0.086214915, payload=None, vector=None, shard_key=None, order_value=None)]

In [40]:
np.argmax(random_embeddings[:1000]@query)

np.int64(845)